# Continuous Hidden Markov Models

This notebook demonstrates continuous HMMs where observations are continuous rather than discrete symbols. We implement Gaussian emissions and mixtures of Gaussians per state following Rabiner (1989) Section VIII.

## GaussianHMM - Single Gaussian per State

For continuous observations, we model emissions using probability density functions. The simplest case uses a single multivariate Gaussian per state.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from hmm.continuous import GaussianHMM
from hmm.algorithms import forward, backward, viterbi, baum_welch

### Creating a GaussianHMM

We specify the number of hidden states and the dimensionality of observations.

In [ ]:
# Create a 2-state GaussianHMM with 1-dimensional observations
hmm = GaussianHMM(
    n_states=2,
    n_features=1,
    means=np.array([[0.0], [10.0]]),
    covars=np.array([[[1.0]], [[1.0]]])
)
print(hmm)

### Emission Probabilities

The emission probability $b_j(o)$ for state $j$ and observation $o$ is given by the Gaussian PDF:

$b_j(o) = \frac{1}{\sqrt{2\pi\sigma_j^2}} \exp\left(-\frac{(o - \mu_j)^2}{2\sigma_j^2}\right)$

In [ ]:
# Calculate emission probabilities
obs_near_0 = np.array([0.0])
obs_near_10 = np.array([10.0])

print(f"P(o=0 | state=0) = {hmm.emission_prob(0, obs_near_0):.4f}")
print(f"P(o=0 | state=1) = {hmm.emission_prob(1, obs_near_0):.4f}")
print(f"P(o=10 | state=0) = {hmm.emission_prob(0, obs_near_10):.4f}")
print(f"P(o=10 | state=1) = {hmm.emission_prob(1, obs_near_10):.4f}")

### Generating Observations

We can generate observation sequences from a trained GaussianHMM.

In [ ]:
np.random.seed(42)

# Generate observations from the model
observations = []
states = []
state = np.random.choice(2, p=hmm.Pi)

for _ in range(100):
    # Sample observation from current state's Gaussian
    obs = hmm.means[state, 0] + np.sqrt(hmm.covars[state, 0, 0]) * np.random.randn()
    observations.append([obs])
    states.append(state)
    # Transition to next state
    state = np.random.choice(2, p=hmm.A[state])

observations = np.array(observations)

plt.figure(figsize=(12, 4))
plt.plot(observations, 'b-', alpha=0.7, label='Observation')
plt.plot(states, 'r--', alpha=0.7, label='Hidden State')
plt.xlabel('Time Step')
plt.ylabel('Observation / State')
plt.title('Generated Observations and Hidden States')
plt.legend()
plt.show()

### Forward Algorithm

The forward algorithm computes the probability of observing a sequence given the model.

In [ ]:
# Compute forward probability
log_prob, alpha, c = forward(hmm, observations, scaling=True)
print(f"Log probability of observation sequence: {log_prob:.4f}")

### Viterbi Decoding

Viterbi finds the most likely sequence of hidden states.

In [ ]:
# Decode the most likely state sequence
q_star, delta, psi = viterbi(hmm, observations, scaling=True)

plt.figure(figsize=(12, 4))
plt.plot(observations, 'b-', alpha=0.7, label='Observation')
plt.plot(states, 'g--', alpha=0.5, label='True State')
plt.plot(q_star, 'r--', alpha=0.7, label='Viterbi Path')
plt.xlabel('Time Step')
plt.ylabel('Observation / State')
plt.title('Viterbi Decoding vs True States')
plt.legend()
plt.show()

accuracy = np.mean(np.array(q_star) == np.array(states)) * 100
print(f"Viterbi accuracy: {accuracy:.1f}%")

### Baum-Welch Training

Training a GaussianHMM involves reestimating the means and covariances using the Baum-Welch algorithm (Rabiner Eq. 53-54).

**Mean update** (Eq. 53):

$\bar{\mu}_j = \frac{\sum_t \gamma_t(j) o_t}{\sum_t \gamma_t(j)}$

**Covariance update** (Eq. 54):

$\bar{\Sigma}_j = \frac{\sum_t \gamma_t(j) (o_t - \mu_j)(o_t - \mu_j)'}{\sum_t \gamma_t(j)}$

In [ ]:
# Create a new GaussianHMM with random initialization
hmm_train = GaussianHMM(n_states=2, n_features=1)

print("Initial means:", hmm_train.means.flatten())
print("Initial covariances:", hmm_train.covars.flatten())

# Train the model
trained_hmm = baum_welch(hmm_train, [observations], epochs=50, verbose=True)

print("\nTrained means:", trained_hmm.means.flatten())
print("Trained covariances:", trained_hmm.covars.flatten())

## MixtureGaussianHMM - Multiple Gaussians per State

For more complex observation densities, we can use a mixture of Gaussians per state.

In [ ]:
from hmm.continuous import MixtureGaussianHMM

### Creating a MixtureGaussianHMM

Each state has multiple Gaussian components with mixture weights.

In [ ]:
# Create a 2-state MixtureGaussianHMM with 2 mixtures per state
mix_hmm = MixtureGaussianHMM(
    n_states=2,
    n_features=1,
    n_mixtures=2,
    means=np.array([[[-2.0], [2.0]], [[8.0], [12.0]]]),
    covars=np.array([[[[1.0]], [[1.0]]], [[[1.0]], [[1.0]]]]),
    weights=np.array([[0.5, 0.5], [0.5, 0.5]])
)
print(mix_hmm)

### Mixture Emission Probability

The emission probability is a weighted sum of Gaussian PDFs:

$b_j(o) = \sum_{k=1}^{M} c_{jk} \mathcal{N}(o | \mu_{jk}, \Sigma_{jk})$

where $c_{jk}$ are the mixture weights.

In [ ]:
# Calculate mixture emission probabilities
obs = np.array([0.0])
probs = mix_hmm.get_emission_probs(obs)
print(f"Mixture emission probabilities at o=0: {probs}")

obs = np.array([10.0])
probs = mix_hmm.get_emission_probs(obs)
print(f"Mixture emission probabilities at o=10: {probs}")

### Training MixtureGaussianHMM

Training involves reestimating mixture weights, means, and covariances (Rabiner Eq. 52-54).

In [ ]:
# Generate training data from mixture model
np.random.seed(42)
train_obs = []
for _ in range(50):
    state = np.random.choice(2, p=mix_hmm.Pi)
    mix_idx = np.random.choice(2, p=mix_hmm.weights[state])
    obs = mix_hmm.means[state, mix_idx, 0] + np.sqrt(mix_hmm.covars[state, mix_idx, 0, 0]) * np.random.randn()
    train_obs.append([obs])
train_obs = np.array(train_obs)

# Train mixture model
mix_train = MixtureGaussianHMM(n_states=2, n_features=1, n_mixtures=2)
trained_mix = baum_welch(mix_train, [train_obs], epochs=50, verbose=False)

print("Initial weights:", mix_train.weights)
print("Trained weights:", trained_mix.weights)
print("\nInitial means:", mix_train.means[:, :, 0])
print("Trained means:", trained_mix.means[:, :, 0])

## Summary

Continuous HMMs extend discrete HMMs by modeling observations as continuous probability distributions rather than discrete symbols. The GaussianHMM uses a single Gaussian per state, while MixtureGaussianHMM uses a mixture of Gaussians for more complex emission densities.

Key algorithms:
- **Forward algorithm**: Computes observation likelihood
- **Viterbi algorithm**: Finds most likely state sequence
- **Baum-Welch**: Reestimates means, covariances, and mixture parameters